# Lesson 4 — Composite scoring: combine judgments with weights you control

Lesson 3 covered asking many questions in one call. This lesson is about combining several of those answers — specifically several `Score`s — into one ranking, using weights that live in your code, not in the model.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/patterns/composite-scoring (click to expand)</summary>

# Composite scoring

> Break a complex judgment into atomic scores, combine with weights you control in code.

Oftentimes we want to rank a set of items based on several criteria at once. Composite scoring is an easy way to think about this: break the judgment into independent dimensions, score each one separately, and combine them with weights you control in code.

## Example: resume screening

Let's imagine you are processing resumes for engineering roles. You want to rank the candidates based on several criteria, and ultimately select the top X candidates for further review.

### Step 1: score each dimension independently

```json
{
  "python_depth": {
    "type": "score",
    "instructions": "How much depth of python experience does this candidate have, based on the supplied resume?",
    "criteria": [
      "No Python experience mentioned",
      "Mentioned but no detail",
      "Used in projects, some specifics",
      "Primary language, multiple projects",
      "Deep expertise: architecture, performance, libraries"
    ]
  },
  "team_leadership": {
    "type": "score",
    "instructions": "How much experience does this candidate have managing or leading engineering teams?",
    "criteria": [
      "No management experience mentioned",
      "Informal mentorship or tech lead role",
      "Led a small team or project",
      "Managed a team with direct reports",
      "Managed multiple teams or an engineering org"
    ]
  },
  "system_design": {
    "type": "score",
    "instructions": "How much experience does this candidate have designing large-scale or distributed systems?",
    "criteria": [
      "No architecture work mentioned",
      "Contributed to design discussions",
      "Designed components of a larger system",
      "Owned architecture of a significant system",
      "Designed systems at scale across multiple domains"
    ]
  },
  "generalist": {
    "type": "score",
    "instructions": "How much evidence is there that this candidate picks up unfamiliar tools, roles, or domains outside their core specialty?",
    "criteria": [
      "Only one domain or role mentioned",
      "Some variety but within a narrow field",
      "Worked across a few different areas or tech stacks",
      "Regularly moved between domains, wore many hats",
      "Track record of ramping up in unfamiliar areas and delivering"
    ]
  }
}
```

### Step 2: combine with weights

Each dimension is normalized to 0–1 and weighted. The weights give you an easy way to adjust the relative importance of each dimension, without losing any of the nuance of the individual scores.

```python
# scoring.py
py      = response.answers["python_depth"].score / 4
lead    = response.answers["team_leadership"].score / 4
arch    = response.answers["system_design"].score / 4
general = response.answers["generalist"].score / 4

# Senior IC
ic_score = (0.40 * py) + (0.10 * lead) + (0.40 * arch) + (0.10 * general)

# Engineering Manager
em_score = (0.15 * py) + (0.40 * lead) + (0.20 * arch) + (0.25 * general)
```

This gives you the ability to rank the candidates based on the composite score. But more importantly, it gives you visibility into how exactly the final score is being calculated. If the highest ranking candidates are not matching your expectations, you can adjust the weights to find the right balance.

</details>

## The core idea

A single `Score` question measures one dimension well — lesson 2 already flagged what happens when you cram several ("punctual and smart and experienced") into one rubric: confidence drops and the score means less. Composite scoring is the fix at the *system* level: ask one `Score` per dimension (they can all go in the same speculative-fan-out call from lesson 3), normalize each to 0–1, then combine them with weights you choose.

The weights never touch the model — they're just arithmetic in your own code. That's the whole advantage: change the weights, get a new ranking instantly, with zero new questions asked.

In [ ]:
import { client } from "../assets/typesafe-client.ts";
import { score } from "npm:@typesafe-ai/sdk";

const resume = `
Senior backend engineer, 6 years. Led a team of 4 engineers rebuilding the
payments pipeline in Python, including designing the event-sourced ledger
and the retry/idempotency layer. Before that, spent two years on the data
platform team writing Go, and a summer doing frontend React work when the
team was short-staffed.
`;

const dimensions = {
  pythonDepth: score(
    "How much depth of python experience does this candidate have, based on the supplied resume?",
    [
      "No Python experience mentioned",
      "Mentioned but no detail",
      "Used in projects, some specifics",
      "Primary language, multiple projects",
      "Deep expertise: architecture, performance, libraries",
    ],
  ),
  teamLeadership: score(
    "How much experience does this candidate have managing or leading engineering teams?",
    [
      "No management experience mentioned",
      "Informal mentorship or tech lead role",
      "Led a small team or project",
      "Managed a team with direct reports",
      "Managed multiple teams or an engineering org",
    ],
  ),
  systemDesign: score(
    "How much experience does this candidate have designing large-scale or distributed systems?",
    [
      "No architecture work mentioned",
      "Contributed to design discussions",
      "Designed components of a larger system",
      "Owned architecture of a significant system",
      "Designed systems at scale across multiple domains",
    ],
  ),
  generalist: score(
    "How much evidence is there that this candidate picks up unfamiliar tools, roles, or domains outside their core specialty?",
    [
      "Only one domain or role mentioned",
      "Some variety but within a narrow field",
      "Worked across a few different areas or tech stacks",
      "Regularly moved between domains, wore many hats",
      "Track record of ramping up in unfamiliar areas and delivering",
    ],
  ),
};

const response = await client.systemOne({
  state: resume,
  questions: dimensions,
});

const { pythonDepth, teamLeadership, systemDesign, generalist } = response.answers;

// Each Score has 5 levels (0-4), so divide by 4 to normalize to 0-1
const py = pythonDepth.score / 4;
const lead = teamLeadership.score / 4;
const arch = systemDesign.score / 4;
const general = generalist.score / 4;

const icScore = 0.4 * py + 0.1 * lead + 0.4 * arch + 0.1 * general;
const emScore = 0.15 * py + 0.4 * lead + 0.2 * arch + 0.25 * general;

console.log({ py, lead, arch, general });
console.log("Senior IC score:", icScore.toFixed(2));
console.log("Engineering Manager score:", emScore.toFixed(2));

## What just happened

Four independent `Score` questions against one resume, normalized to 0–1, then combined two different ways using the docs' own weight profiles. The Senior IC weighting leans on Python depth and system design (0.40 each); the Engineering Manager weighting leans on team leadership (0.40). Same four scores, two different rankings — because the weights, not the model, decide what matters for each role.

**Try it yourself:** predict which of `icScore` and `emScore` will come out higher for this resume (it's leadership-heavy but also has real architecture work) before running it. Then edit the resume to describe a candidate with zero management experience and deep distributed-systems work, and predict how the two scores move relative to each other.

## Next up

Next lesson: Function calling — mapping a natural-language request to a typed function call with closed-set arguments, each scored with its own confidence.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.